# Make LLM Helpers Auditable and Reproducible

In this lesson, we keep the same helper pattern from earlier modules, then add logging and versioning configs so every run is reviewable and reproducible.


## 1 — Setup

The completed notebook includes saved outputs so you can review the expected result without my local `.env` file. To rerun the Gemini cells, create your own `.env` file in the project root with `GEMINI_API_KEY=your-gemini-api-key-here`.


We use the same setup pattern as the first helper notebook: install libraries, load credentials, and read the dataset.


In [ ]:
#%pip install -qq google-genai pandas scikit-learn matplotlib seaborn python-dotenv

In [17]:
import os
import json
from datetime import datetime

from google import genai
from dotenv import load_dotenv
import pandas as pd

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)

In [18]:
df = pd.read_csv("../../data/hr_analytics.csv")


## 2 — The LLM Helper Function from Module 0

In [19]:
SYSTEM_PROMPT = (
    "Write pandas code using the existing DataFrame variable `df`. "
    "Use up-to-date pandas 2.x and Python 3.10+ syntax. Avoid deprecated arguments or methods. "
    "Store the final result in `result_df`. "
    "Do NOT create a new DataFrame from scratch. "
    "Do NOT include import statements. "
    "Return ONLY executable Python code, no explanations."
)

In [20]:
def eda_helper(question, frame=pd.DataFrame, show_code: bool = False):
    """Ask a plain-English question about df; get back a DataFrame."""
    prompt = f"Columns: {list(df.columns)}\nQuestion: {question}"

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config={
            "temperature": 0.0,
            "seed": 42,
            "system_instruction": SYSTEM_PROMPT,
        },
    )

    code = response.text
    if "```" in code:
        code = code.split("```")[1].replace("python", "").strip()

    # Optionally print the generated code before executing it.
    # This lets you inspect what the model wrote and verify that it matches your intent.
    if show_code:
        print("--- Generated Code ---")
        print(code)
        print("---\n")

    env = {"pd": pd, "df": frame.copy()}
    exec(code, env, env)

    result = env.get("result_df")
    if result is None:
        raise RuntimeError("Generated code did not assign `result_df`.")
    return result


## 3 — Enterprise Essentials: Auditability, Reuse, Versioning

A production helper needs more than a correct answer. It needs:

| Concern | Why it matters |
|---|---|
| **Auditability** | Regulators and stakeholders must see *what code* produced the result |
| **Reuse** | Anyone should be able to re-run the same analysis and get the same results |
| **Versioning** | Track which model and prompt version produced each result |

Below we wrap the same helper with a lightweight audit log. 

### 3.1 — Keep a Simple Log

Once you can see the code, the next natural question is: *can I go back and check what ran earlier?*

A lightweight log with just a list of `(timestamp, question, code)` entries that gives you a basic audit trail without any extra complexity.

In [21]:
log = []

LOG_CONFIG = {
    "version": "1.0.0",
    "model": "gemini-2.5-flash",
    "system_instruction": SYSTEM_PROMPT,
}


def eda_helper(question, helper_config=LOG_CONFIG, frame=None, show_code=False):
    if frame is None:
        frame = df

    prompt = f"Columns: {list(frame.columns)}\nQuestion: {question}"

    response = client.models.generate_content(
        model=helper_config["model"],
        contents=prompt,
        config={
            "temperature": 0.0,
            "seed": 42,
            "system_instruction": helper_config["system_instruction"],
        },
    )

    code = response.text
    if "```" in code:
        code = code.split("```")[1].replace("python", "").strip()

    if show_code:
        print("--- Generated Code ---")
        print(code)
        print("---")

    entry = {
        "timestamp": datetime.now().isoformat(),
        "agent_version": helper_config["version"],
        "model": helper_config["model"],
        "question": question,
        "generated_code": code,
        "status": "pending",
    }

    try:
        env = {"pd": pd, "df": frame.copy()}
        exec(code, env, env)
        result = env.get("result_df")
        if result is None:
            raise RuntimeError("Generated code did not assign `result_df`.")
        entry["status"] = "success"
        entry["result_shape"] = str(result.shape)
    except Exception as e:
        entry["status"] = "error"
        entry["error"] = str(e)
        log.append(entry)
        print(f"Execution failed: {e}")
        return None

    log.append(entry)
    return result

In [22]:
eda_helper("What are the top 10 highest paid employees?", frame=df, show_code=True)

--- Generated Code ---
result_df = df.nlargest(10, 'MonthlyIncome')
---


,Employee ID,age,gender,department,department_code,JobTitle,job_level,Education,MonthlyIncome,monthly_rate,...,satisfaction_score,environment_satisfaction,Attrition,OverTime,distance_from_home,training_hours_last_year,num_companies_worked,manager_rating,work_life_balance,last_promotion_date
2306,3307,54,Male,Engineering,ENG-02,ML Engineer,5,Master's,20659,23089,...,3 - High,4,No,No,12.0,20.0,0,3.0,High,2017-07-31
1233,2234,38,Male,Engineering,ENG-02,Staff Engineer,5,PhD,20396,24862,...,2 - Medium,3,No,No,2.0,12.0,1,4.0,Very High,2022-07-02
2461,3462,43,Female,Engineering,ENG-02,ML Engineer,5,Master's,20054,26996,...,3 - High,4,No,No,7.0,42.0,0,4.0,Very High,2007-12-03
4648,5649,24,Female,Engineering,ENG-02,Staff Engineer,5,PhD,19422,22028,...,3 - High,2,No,No,3.0,16.0,3,5.0,Very High,11/28/2024
3856,4857,42,Female,Engineering,ENG-02,Engineering Manager,5,Master's,18350,24309,...,3 - High,4,No,No,5.0,21.0,3,4.0,Very High,08/22/2021
2986,3987,40,Male,Engineering,ENG-02,ML Engineer,5,Bachelor's,18214,24025,...,3 - High,4,No,No,7.0,33.0,0,5.0,Very High,2023-10-12
1089,2090,50,Female,Engineering,ENG-02,DevOps Engineer,5,Bachelor's,17614,23839,...,2 - Medium,3,No,No,3.0,32.0,2,3.0,Medium,2018-03-12
2229,3230,32,Female,Sales,SAL-03,Sales Director,5,PhD,17542,22584,...,4 - Very High,4,No,No,6.0,26.0,0,3.0,High,10/15/2022
3836,4837,46,Female,Sales,SAL-03,Sales Director,5,Bachelor's,17511,21074,...,4 - Very High,3,Yes,Yes,5.0,20.0,2,4.0,Very High,04/05/2021
643,1644,39,Female,Engineering,ENG-02,Staff Engineer,5,Bachelor's,17459,22266,...,2 - Medium,2,Yes,No,19.0,30.0,0,3.0,Very High,2020-04-25


In [23]:
log

[{'timestamp': '2026-04-19T10:19:03.990238',
  'agent_version': '1.0.0',
  'model': 'gemini-2.5-flash',
  'question': 'What are the top 10 highest paid employees?',
  'generated_code': "result_df = df.nlargest(10, 'MonthlyIncome')",
  'status': 'success',
  'result_shape': '(10, 25)'}]

### 3.2 — Inspect the Audit Log

In [24]:
audit_df = pd.DataFrame(log)
print(f"Queries logged : {len(audit_df)}")
if len(audit_df):
    print(f"Successful     : {(audit_df['status'] == 'success').sum()}")
    print(f"Failed         : {(audit_df['status'] == 'error').sum()}")
    display(audit_df[["timestamp", "question", "status", "model", "generated_code"]])
else:
    print("No log entries yet. Run eda_helper(...) first.")

Queries logged : 1
Successful     : 1
Failed         : 0


,timestamp,question,status,model,generated_code
0,2026-04-19T10:19:03.990238,What are the top 10 highest paid employees?,success,gemini-2.5-flash,"result_df = df.nlargest(10, 'MonthlyIncome')"


### 3.3 — Replay Without an API Call

Since the generated code is stored, you can re-run any past analysis offline — no LLM call required. This saves cost and guarantees identical logic.

In [25]:
def replay(query_index):
    """Re-execute a logged query without calling the LLM."""
    entry = log[query_index]
    print(f"Replaying : {entry['question']}")
    print(f"Originally: {entry['timestamp']}")
    print(f"Code:{entry['generated_code']}")

    env = {"pd": pd, "df": df.copy()}
    exec(entry["generated_code"], env, env)
    result = env.get("result_df")
    if result is None:
        raise RuntimeError("Generated code did not assign `result_df`.")
    return result


replay(0)

Replaying : What are the top 10 highest paid employees?
Originally: 2026-04-19T10:19:03.990238
Code:result_df = df.nlargest(10, 'MonthlyIncome')


,Employee ID,age,gender,department,department_code,JobTitle,job_level,Education,MonthlyIncome,monthly_rate,...,satisfaction_score,environment_satisfaction,Attrition,OverTime,distance_from_home,training_hours_last_year,num_companies_worked,manager_rating,work_life_balance,last_promotion_date
2306,3307,54,Male,Engineering,ENG-02,ML Engineer,5,Master's,20659,23089,...,3 - High,4,No,No,12.0,20.0,0,3.0,High,2017-07-31
1233,2234,38,Male,Engineering,ENG-02,Staff Engineer,5,PhD,20396,24862,...,2 - Medium,3,No,No,2.0,12.0,1,4.0,Very High,2022-07-02
2461,3462,43,Female,Engineering,ENG-02,ML Engineer,5,Master's,20054,26996,...,3 - High,4,No,No,7.0,42.0,0,4.0,Very High,2007-12-03
4648,5649,24,Female,Engineering,ENG-02,Staff Engineer,5,PhD,19422,22028,...,3 - High,2,No,No,3.0,16.0,3,5.0,Very High,11/28/2024
3856,4857,42,Female,Engineering,ENG-02,Engineering Manager,5,Master's,18350,24309,...,3 - High,4,No,No,5.0,21.0,3,4.0,Very High,08/22/2021
2986,3987,40,Male,Engineering,ENG-02,ML Engineer,5,Bachelor's,18214,24025,...,3 - High,4,No,No,7.0,33.0,0,5.0,Very High,2023-10-12
1089,2090,50,Female,Engineering,ENG-02,DevOps Engineer,5,Bachelor's,17614,23839,...,2 - Medium,3,No,No,3.0,32.0,2,3.0,Medium,2018-03-12
2229,3230,32,Female,Sales,SAL-03,Sales Director,5,PhD,17542,22584,...,4 - Very High,4,No,No,6.0,26.0,0,3.0,High,10/15/2022
3836,4837,46,Female,Sales,SAL-03,Sales Director,5,Bachelor's,17511,21074,...,4 - Very High,3,Yes,Yes,5.0,20.0,2,4.0,Very High,04/05/2021
643,1644,39,Female,Engineering,ENG-02,Staff Engineer,5,Bachelor's,17459,22266,...,2 - Medium,2,Yes,No,19.0,30.0,0,3.0,Very High,2020-04-25


### 3.4 — Version & Compare Models

Change the model or prompt and the log tells you exactly which config produced each result.

In [26]:
# CONFIG_V2 shows how you'd test a different model — swap the model name to compare outputs.
CONFIG_V2 = {
    **LOG_CONFIG,
    "version": "2.0.0",
    "model": "gemini-2.0-flash",
}

question = "Who are the top 5 employees closest to retirement age (65) who have never been promoted?"

result_v1 = eda_helper(question, helper_config=LOG_CONFIG)
result_v2 = eda_helper(question, helper_config=CONFIG_V2)

print(f"=== v1 ({LOG_CONFIG['model']}) ===")
print(result_v1)
print(f"\n=== v2 ({CONFIG_V2['model']}) ===")
print(result_v2)

=== v1 (gemini-2.5-flash) ===
      Employee ID  age gender       department department_code  \
4734         5735   48   Male       Operations          OPS-06   
3201         4202   48   Male        Marketing          MKT-04   
4328         5329   46   Male  Human Resources           HR-01   
3734         4735   46   Male      Engineering          ENG-02   
2599         3600   45   Male      Engineering          ENG-02   

                  JobTitle  job_level    Education  MonthlyIncome  \
4734  Supply Chain Analyst          1  High School           4475   
3201     Marketing Analyst          1          PhD           5211   
4328            HR Manager          3  High School           6965   
3734   Engineering Manager          2  High School           7269   
2599        Staff Engineer          2   Bachelor's           8198   

      monthly_rate  ...  environment_satisfaction  Attrition  OverTime  \
4734          6320  ...                         3         No        No   
3201      

### 3.5 — Export the Audit Log

In [27]:
with open("log.json", "w") as f:
    json.dump(log, f, indent=2)
print("\u2713 Audit log exported to log.json")

✓ Audit log exported to log.json
